In [14]:
%cd /workspace/EBES

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from pathlib import Path
import optuna 
from ebes.pipeline.utils import optuna_df
from optuna.trial import TrialState

/workspace/EBES


/usr/local/lib/python3.10/dist-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [15]:
def get_run(number, specify="best", rewrite=False):
    path = Path(f"log/{dataset}/{method}/optuna/{number}")
    print(pd.read_csv(path / "results.csv"))
    print((path / "params.txt").read_text())
    save_path = Path(f"configs/specify/{dataset}/{method}")
    save_path.mkdir(parents=True, exist_ok=True)
    save_path = (save_path / f"{specify}.yaml")
    if not rewrite:
        assert not save_path.exists()
    save_path.write_text((path / "params.txt").read_text())

def prepare_data(dataset, method):
    path = Path(f"log/{dataset}/{method}/optuna")
    df, study = optuna_df(path)
    value_pack = ["values_0", "values_1", "values_2", "values_3"]
    col_to_drop = ["datetime_start", "datetime_complete", "system_attrs_fixed_params", "state", *value_pack]
    col_params = [*value_pack, "duration"] + [col for col in df if "params_" in col]
    col_user = [*value_pack, "duration"] + [col for col in df if "user" in col]
    df["duration"] = df["duration"].dt.total_seconds()
    return df, study, col_user, col_params

In [16]:
dataset = "age"
method = "coles"
df, study, col_user, col_params = prepare_data(dataset, method)

print(df.columns)
core_param = "values_0"
print(df.shape, df[~df[core_param].isna()].shape)
test_cols = [col for col in col_user if ("test" in col)]
df[~df[core_param].isna()].sort_values(core_param, ascending=False).iloc[:10, ][[core_param] + test_cols]

Index(['values_0', 'values_1', 'values_2', 'values_3', 'datetime_start',
       'datetime_complete', 'duration',
       'params_data.loaders.unsupervised_train.batch_size',
       'params_model.aggregation.name',
       'params_model.emb_head.params.out_features',
       'params_model.encoder.params.hidden_size',
       'params_model.encoder.params.num_layers',
       'params_model.preprocess.params.cat_emb_dim',
       'params_model.preprocess.params.num_emb_dim',
       'params_model.preprocess.params.num_norm',
       'params_model.preprocess.params.time_process',
       'params_optimizer.params.lr', 'params_optimizer.params.weight_decay',
       'user_attrs_loss_mean', 'user_attrs_loss_std',
       'user_attrs_memory_after_mean', 'user_attrs_memory_after_std',
       'user_attrs_target__age__global__accuracy+f1_macro_mean',
       'user_attrs_target__age__global__accuracy+f1_macro_std',
       'user_attrs_target__anomaly__global__roc_auc+f1_macro+accuracy_mean',
       'user_attrs_

/workspace/EBES/ebes/pipeline/utils.py:168: ExperimentalWarning: JournalStorage is experimental (supported from v3.1.0). The interface can change in the future.
  storage = JournalStorage(JournalFileStorage(f"{path}/study.log"))


,values_0
88,0.493754
87,0.467951
89,0.463799
92,0.456624
104,0.456294
52,0.437556
58,0.434059
51,0.426910
91,0.409454
94,0.403481


In [ ]:
#get_run(33, specify="best_Specific_name")

In [18]:
failed = df[(df["state"] != "COMPLETE") | (df[col_user].isna().any(axis=1))][col_user].index
print(failed)
for fail in df[(df["state"] != "COMPLETE") | (df[col_user].isna().any(axis=1))][col_user].index:
    error_path = Path(f"/home/dev/24/es-bench/log/{dataset}/{method}/optuna/{fail}/ERROR.txt")
    if error_path.exists():
        error = error_path.read_text()
        print(fail, error.split("\n")[-2])
    else:
        print(df.loc[fail])

Index([  1,   9,  29,  31,  34,  43,  48,  54,  56,  59,  60,  61,  64,  66,
        69,  70,  72,  74,  76,  77,  96,  98, 100],
      dtype='int64')
values_0                                                                                     NaN
values_1                                                                                     NaN
values_2                                                                                     NaN
values_3                                                                                     NaN
datetime_start                                                        2026-03-17 22:08:51.080852
datetime_complete                                                     2026-03-17 22:08:55.000409
duration                                                                                3.919557
params_data.loaders.unsupervised_train.batch_size                                            512
params_model.aggregation.name                                            

In [20]:
optuna.visualization.plot_optimization_history(study, target = lambda t: t.values[int(core_param[-1])])

/usr/local/lib/python3.10/dist-packages/optuna/visualization/_utils.py:67: UserWarning: `target` is specified, but `target_name` is the default value, 'Objective Value'.
  warnings.warn(


### Params influence

In [25]:
target_objective_index = int(core_param[-1])
trials = study.trials
trials = [trial for trial in trials if trial.state == TrialState.COMPLETE]
plotted_trials = sorted(trials, key=lambda t: t.values[target_objective_index])[:]
plotted_study = optuna.create_study()
for trial in plotted_trials:
    # Создаем "чистую" копию триала только с одним нужным значением
    single_value_trial = optuna.trial.create_trial(
        params=trial.params,
        distributions=trial.distributions,
        value=trial.values[target_objective_index], # Берем только одно значение
    )
    plotted_study.add_trial(single_value_trial)

[I 2026-03-23 19:28:00,874] A new study created in memory with name: no-name-49039f34-cf74-4c51-9077-fc5d7c338e76


In [26]:
target = None #lambda t: (t.user_attrs["memory_after_mean"])
target_name = "value"
fig = optuna.visualization.plot_param_importances(plotted_study, target=target, target_name=target_name)
print(fig._data[0]["x"][::-1])
print(fig._data[0]["y"][::-1])
take = 7
params = fig._data[0]["y"][-take:]
not_imp = list(set([col.replace("params_", "") for col in col_params]) - set(params) - {"duration", "value", "system_attrs_fixed_params"})
fig

[0.2856147588625646, 0.14444341950581097, 0.1221969096894113, 0.12041627520256928, 0.10759298315252187, 0.06745356924722112, 0.06210268318854591, 0.034364507913194176, 0.02607855609172666, 0.017644058209983965, 0.012092278936450278]
['optimizer.params.lr', 'optimizer.params.weight_decay', 'model.encoder.params.hidden_size', 'model.preprocess.params.cat_emb_dim', 'model.encoder.params.num_layers', 'model.preprocess.params.num_emb_dim', 'model.emb_head.params.out_features', 'data.loaders.unsupervised_train.batch_size', 'model.aggregation.name', 'model.preprocess.params.time_process', 'model.preprocess.params.num_norm']


In [27]:
params

['model.emb_head.params.out_features',
 'model.preprocess.params.num_emb_dim',
 'model.encoder.params.num_layers',
 'model.preprocess.params.cat_emb_dim',
 'model.encoder.params.hidden_size',
 'optimizer.params.weight_decay',
 'optimizer.params.lr']